# 🗄️ Clase 2: Bases de Datos y Limpieza de Datos

**Curso: Modelos de AI en Negocios (MAIN) - IA Aplicada a la Industria**
Universidad de Santiago de Chile · Departamento de Ingeniería Industrial

---

## 🏭 Contexto del caso

Somos analistas de datos en una empresa industrial. El equipo de calidad nos ha entregado un dataset real extraído de **Kaggle** con **1.000 registros de defectos detectados en procesos de manufactura**, incluyendo el tipo de defecto, su severidad, la fecha de detección, la ubicación en el producto, el método de inspección utilizado y el costo de reparación.

**El problema:** los datos tienen errores — valores faltantes, fechas guardadas como texto y costos imposibles — que impiden usarlos directamente para tomar decisiones.

**Nuestro objetivo:**
1. Detectar y corregir los problemas de calidad del dataset
2. Calcular el **costo promedio de reparación** por tipo de defecto y severidad
3. Identificar **qué tipo de defecto es más costoso** y **qué método de inspección lo detecta mejor**
4. Exportar un CSV limpio y gráficos listos para presentar a gerencia

---

## 📋 Diccionario de datos (columnas del CSV)

| Columna | Descripción | Tipo esperado |
|---|---|---|
| `defect_id` | Identificador único del defecto | Entero |
| `product_id` | Identificador del producto afectado | Entero |
| `defect_type` | Tipo de defecto: Structural, Functional, Cosmetic | Texto categórico |
| `defect_date` | Fecha de detección del defecto | Fecha (guardada como texto — dato sucio) |
| `defect_location` | Ubicación en el producto: Surface, Component, Internal | Texto categórico |
| `severity` | Severidad: Minor, Moderate, Critical | Texto / **NaN** (dato sucio) |
| `inspection_method` | Método de inspección: Visual, Automated, Manual | Texto / **NaN** (dato sucio) |
| `repair_cost` | Costo de reparación en moneda local | Float (algunos valores **negativos** — error de entrada) |

---

## ⚠️ Problemas conocidos en los datos (lo que vamos a solucionar)

- **Valores nulos** en `severity` (~10%) y `inspection_method` (~8%) — campos no siempre registrados
- **Fechas como texto** en `defect_date` en lugar de tipo fecha/datetime
- **Costos negativos** en `repair_cost` (~3%) — errores de ingreso de datos

> 💡 **Nota sobre el dataset:** El CSV fue descargado de [Kaggle — Manufacturing Defects](https://www.kaggle.com/datasets/fahmidachowdhury/manufacturing-defects). Es un dataset sintético pero realista, diseñado para análisis de control de calidad. Los problemas de calidad de datos fueron **inyectados intencionalmente** en el Paso 0 para simular condiciones reales de manufactura.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  ¿QUÉ ES UN "import"?
#  Python solo viene con herramientas básicas. Las "librerías" son
#  paquetes de funciones que alguien ya programó y nosotros podemos
#  reutilizar gratis sin tener que programarlas desde cero.
#
#  Sintaxis:  import <librería> as <alias>
#    → "as np" significa: usaré el apodo "np" para no escribir
#       "numpy" completo cada vez que la necesite.
#  Esta es la convención universal de la comunidad Python.
# ══════════════════════════════════════════════════════════════════

import numpy as np               # numpy  → cálculos numéricos y generación de índices aleatorios
import pandas as pd              # pandas → nuestra herramienta principal para manejar tablas de datos
import matplotlib.pyplot as plt  # matplotlib → para crear gráficos y visualizaciones
import matplotlib.ticker as mticker  # herramienta auxiliar de matplotlib para formatear ejes

# La función print() muestra texto en pantalla.
print('✅ Librerías cargadas correctamente')
print()
print('  numpy  (np)  → operaciones numéricas y generación de índices aleatorios')
print('  pandas (pd)  → leer, explorar, limpiar y transformar tablas de datos (DataFrames)')
print('  matplotlib   → crear gráficos de barras, líneas y visualizaciones')

---
## 📂 Paso 0: Cargar los datos e inyectar problemas de calidad

El archivo `defects_data.csv` fue descargado desde Kaggle. Para obtenerlo:
- **Opción A (Colab):** Subir el archivo manualmente a la sesión y leerlo con `pd.read_csv('defects_data.csv')`
- **Opción B (con kagglehub):** Ejecutar `kagglehub.dataset_download("fahmidachowdhury/manufacturing-defects")`

Una vez cargado el CSV limpio, **inyectamos nulos y errores** intencionalmente para simular los problemas que encontraríamos en datos reales de manufactura.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CONCEPTOS CLAVE DE ESTA CELDA
#  ─────────────────────────────────────────────────────────────────
#  DataFrame (df):
#    Un DataFrame es simplemente una TABLA de datos, igual a una
#    hoja de Excel, pero dentro de Python. Tiene filas y columnas
#    con nombres. La variable "df" es la que usaremos para guardar
#    y manipular nuestra tabla a lo largo del notebook.
#
#  pd.read_csv('archivo.csv'):
#    Lee un archivo CSV (valores separados por comas) y lo convierte
#    en un DataFrame. Es como abrir un Excel, pero en Python.
#    El archivo debe estar en la misma carpeta que este notebook.
# ══════════════════════════════════════════════════════════════════

# Cargar el archivo CSV desde Kaggle y guardarlo en la variable "df"
# "df" es el nombre convencional que se usa para un DataFrame en Python
df = pd.read_csv('defects_data.csv')

# df.shape → devuelve una tupla (filas, columnas)
# df.shape[0] → número de filas
# df.shape[1] → número de columnas
# Usamos f-strings (f'...') para insertar variables dentro de un texto: {variable}
print(f'Dataset original cargado: {df.shape[0]} filas, {df.shape[1]} columnas')

# df.columns → lista con los nombres de todas las columnas
# list() convierte esa lista al formato estándar de Python
print(f'Columnas: {list(df.columns)}')

# df.isnull() → crea una tabla True/False: True donde hay un valor vacío (NaN)
# .sum()      → suma los True (True = 1, False = 0) por columna
# .sum()      → suma total de todas las columnas → total de celdas vacías
print(f'Nulos originales: {df.isnull().sum().sum()} (el CSV de Kaggle viene limpio)')

print()
print('─' * 60)
print('  PASO 0b: Inyectando problemas de calidad para el ejercicio')
print('─' * 60)

# ─── INYECCIÓN DE PROBLEMAS DE CALIDAD ──────────────────────────
# En la industria real, los datos llegan con estos errores porque
# el sistema de registro falla o los operadores no llenan todos los campos.
# Aquí los simulamos intencionalmente para practicar la limpieza.

# np.random.seed(42):
#   Fija la "semilla" del generador de números aleatorios.
#   Con la misma semilla, SIEMPRE obtenemos los mismos números aleatorios.
#   → Esto garantiza que todos los estudiantes tengan los mismos datos sucios.
np.random.seed(42)

# len(df) → devuelve el número total de filas del DataFrame
n = len(df)

# np.random.choice(n, size=X, replace=False):
#   Elige al azar X números diferentes entre 0 y n-1.
#   Son los ÍNDICES (número de fila) que vamos a modificar.
#   → int(n * 0.10) = 10% de las filas → aproximadamente 100 filas
# df.loc[índices, 'columna']:
#   "loc" significa "location" → accede a filas específicas por su índice.
#   Aquí le decimos: "en esas filas, en la columna 'severity', pon NaN"
# np.nan → el valor especial de Python que representa "dato faltante" (celda vacía)
idx_null_sev = np.random.choice(n, size=int(n * 0.10), replace=False)
df.loc[idx_null_sev, 'severity'] = np.nan   # Problema 1: severidad no registrada

idx_null_insp = np.random.choice(n, size=int(n * 0.08), replace=False)
df.loc[idx_null_insp, 'inspection_method'] = np.nan  # Problema 2: método no registrado

# Costos negativos → error de signo al teclear (ej: -250 en vez de 250)
# El signo menos (-) delante de una variable invierte su signo: positivo → negativo
idx_neg_cost = np.random.choice(n, size=int(n * 0.03), replace=False)
df.loc[idx_neg_cost, 'repair_cost'] = -df.loc[idx_neg_cost, 'repair_cost']  # Problema 3

print('\n⚠️  Problemas inyectados para el ejercicio:')
# df["columna"].isnull() → serie True/False; .sum() cuenta los True (los nulos)
print(f'   - Nulos en "severity":          {df["severity"].isnull().sum()} registros')
print(f'   - Nulos en "inspection_method": {df["inspection_method"].isnull().sum()} registros')
# (df["repair_cost"] < 0) → serie True/False; .sum() cuenta los negativos
print(f'   - Costos negativos:             {(df["repair_cost"] < 0).sum()} registros')
print(f'\nDataset listo para limpiar: {df.shape[0]} filas, {df.shape[1]} columnas')

---
## 🔍 Paso 1: Exploración inicial — ¿Qué tenemos?

Antes de limpiar cualquier cosa, necesitamos **entender la estructura del dataset**:
- ¿Cuántas filas y columnas tiene?
- ¿Qué tipo de datos hay en cada columna?
- ¿Hay valores faltantes?
- ¿Los números tienen sentido a primera vista?

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  df.head(n) → muestra las primeras n filas del DataFrame.
#              Si no le pasas ningún número, muestra las primeras 5.
#  Es el primer comando que se ejecuta al explorar datos nuevos:
#  nos da una "foto" rápida de cómo se ven los datos.
# ══════════════════════════════════════════════════════════════════
print('=== Primeras 5 filas del dataset ===')
df.head()
# NOTA: en Jupyter/Colab, la ÚLTIMA línea de una celda que sea un
# DataFrame se muestra automáticamente como tabla. No necesita print().

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  df.shape → tupla (filas, columnas). Ej: (1000, 8)
#
#  df.info() → resumen técnico del DataFrame. Muestra:
#    - Número total de filas
#    - Nombre de cada columna
#    - Cuántos valores NO nulos tiene cada columna
#    - El tipo de dato de cada columna:
#        int64   → número entero (ej: 1, 42, 1000)
#        float64 → número decimal (ej: 3.14, 250.75)
#        object  → texto o cadena de caracteres (ej: "Structural")
#        datetime64 → fecha y hora (ej: 2023-05-12)
#  Es fundamental para saber QUÉ contiene cada columna antes de analizar.
# ══════════════════════════════════════════════════════════════════
print('=== Información general del dataset ===')
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print()
df.info()
# Atención: si "Non-Null Count" < total de filas → hay nulos en esa columna

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  df.describe() → genera automáticamente estadísticas descriptivas
#  para todas las columnas NUMÉRICAS del DataFrame:
#
#    count  → cuántos valores no nulos hay
#    mean   → promedio aritmético
#    std    → desviación estándar (qué tan dispersos están los datos)
#    min    → valor mínimo
#    25%    → primer cuartil (25% de los datos están bajo este valor)
#    50%    → mediana (valor del medio)
#    75%    → tercer cuartil (75% de los datos están bajo este valor)
#    max    → valor máximo
#
#  ¿Para qué sirve? Para detectar anomalías rápidamente:
#  si el mínimo de "repair_cost" es negativo → hay un problema.
# ══════════════════════════════════════════════════════════════════
print('=== Estadísticas descriptivas (columnas numéricas) ===')
df.describe()
# Observa el valor "min" de repair_cost → debería ser negativo después del Paso 0

---
## 🚨 Paso 2: Detectar problemas de calidad

Ahora que conocemos la estructura, vamos a **identificar los tres problemas** que sabemos que existen:
1. Valores nulos en `severity` e `inspection_method`
2. Costos de reparación negativos (imposibles)
3. Columna `defect_date` almacenada como texto en vez de fecha

In [ ]:
# Contar valores nulos por columna
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(1)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': porcentaje_nulos
})

print('=== Valores nulos por columna ===')
print(resumen_nulos[resumen_nulos['Nulos'] > 0])
print(f'\nTotal de celdas vacías: {nulos.sum()}')

In [ ]:
# Detectar registros con costo de reparación negativo (imposible en la realidad)
costos_negativos = df[df['repair_cost'] < 0]

print(f'=== Registros con costo negativo: {len(costos_negativos)} ===')
print(costos_negativos[['defect_id', 'defect_type', 'severity', 'repair_cost']].head(10))

In [ ]:
# Verificar si 'defect_date' es texto o tipo fecha real
print(f'Tipo de dato de la columna defect_date: {df["defect_date"].dtype}')
print(f'Ejemplos de valores: {df["defect_date"].head(5).tolist()}')
print()
print('⚠️  La columna es tipo "object" (texto). Debemos convertirla a datetime para poder'
      ' filtrar por fecha y calcular tendencias temporales.')

---
## 🧹 Paso 3: Limpieza de datos

Ahora que sabemos **qué está mal**, vamos a corregirlo sistemáticamente. Este es el corazón del trabajo de un analista de datos.

El orden importa: primero convertimos tipos, luego tratamos nulos, y finalmente eliminamos registros inválidos.

In [ ]:
# Hacer una copia para no modificar el original (buena práctica)
df_limpio = df.copy()

# ─── CORRECCIÓN 1: Convertir 'defect_date' de texto a tipo fecha ───
df_limpio['defect_date'] = pd.to_datetime(df_limpio['defect_date'])

print(f'✅ Tipo de dato después de la conversión: {df_limpio["defect_date"].dtype}')
print(f'   Rango de fechas: {df_limpio["defect_date"].min().date()} → {df_limpio["defect_date"].max().date()}')

In [ ]:
# ─── CORRECCIÓN 2: Rellenar nulos en columnas categóricas ───
# Estrategia: reemplazar con una categoría centinela para no perder filas
df_limpio['severity']          = df_limpio['severity'].fillna('Desconocida')
df_limpio['inspection_method'] = df_limpio['inspection_method'].fillna('No registrado')

print('✅ Nulos en columnas categóricas reemplazados:')
print(f'   severity — categorías disponibles:          {sorted(df_limpio["severity"].unique())}')
print(f'   inspection_method — categorías disponibles: {sorted(df_limpio["inspection_method"].unique())}')
print(f'\nNulos restantes en el DataFrame: {df_limpio.isnull().sum().sum()}')

In [ ]:
# ─── CORRECCIÓN 3: Eliminar filas con costo de reparación negativo ───
# Un costo negativo es un error de ingreso de datos — no se puede "corregir" sin información adicional
filas_antes  = len(df_limpio)
df_limpio    = df_limpio[df_limpio['repair_cost'] >= 0].reset_index(drop=True)
filas_despues = len(df_limpio)

print(f'✅ Filas eliminadas por costo negativo: {filas_antes - filas_despues}')
print(f'   Filas restantes: {filas_despues}')

In [ ]:
# Resumen del proceso de limpieza
print('╔══════════════════════════════════════════════════════╗')
print('║          RESUMEN DEL PROCESO DE LIMPIEZA            ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Filas originales (con errores):    {len(df):>5}             ║')
print(f'║  Filas en dataset limpio:           {len(df_limpio):>5}             ║')
print(f'║  Filas eliminadas (costos neg.):    {len(df) - len(df_limpio):>5}             ║')
print('╠══════════════════════════════════════════════════════╣')
print('║  Correcciones realizadas:                           ║')
print('║    1. defect_date → convertida a datetime           ║')
print('║    2. severity → nulos → "Desconocida"              ║')
print('║    3. inspection_method → nulos → "No registrado"  ║')
print('║    4. repair_cost < 0 → filas eliminadas            ║')
print('╚══════════════════════════════════════════════════════╝')

# Tipos de dato finales
print('\nTipos de dato del DataFrame limpio:')
print(df_limpio.dtypes)

---
## 📊 Paso 4: Calcular métricas de negocio

Con los datos limpios, ahora podemos responder la **pregunta de negocio**:

> "¿Qué tipo de defecto genera mayor costo de reparación, y qué método de inspección es más efectivo para detectarlo?"

Calcularemos:
1. Distribución de defectos por tipo y por severidad
2. Costo promedio de reparación por tipo de defecto
3. Costo promedio de reparación por severidad
4. Cantidad de defectos detectados por cada método de inspección

In [ ]:
# ─── Métrica 1: Distribución de defectos por tipo ───
conteo_tipo = df_limpio['defect_type'].value_counts()
pct_tipo    = (conteo_tipo / conteo_tipo.sum() * 100).round(1)

print('=== Distribución por tipo de defecto ===')
for tipo, cnt in conteo_tipo.items():
    print(f'  {tipo:12s}: {cnt:4d} defectos  ({pct_tipo[tipo]}%)')

print()

# ─── Métrica 2: Distribución por severidad ───
conteo_sev = df_limpio['severity'].value_counts()
pct_sev    = (conteo_sev / conteo_sev.sum() * 100).round(1)

print('=== Distribución por severidad ===')
for sev, cnt in conteo_sev.items():
    print(f'  {sev:13s}: {cnt:4d} defectos  ({pct_sev[sev]}%)')

In [ ]:
# ─── Métrica 3: Costo promedio de reparación por tipo de defecto ───
costo_por_tipo = (
    df_limpio.groupby('defect_type')['repair_cost']
    .agg(n_defectos='count', costo_promedio='mean', costo_total='sum')
    .round(2)
    .sort_values('costo_promedio', ascending=False)
)

print('=== Costo de reparación por tipo de defecto ===')
print(costo_por_tipo.to_string())
print(f'\n➡  El tipo más costoso es: {costo_por_tipo["costo_promedio"].idxmax()}'
      f'  (${costo_por_tipo["costo_promedio"].max():.2f} promedio)')

print()

# ─── Métrica 4: Costo promedio por severidad ───
costo_por_sev = (
    df_limpio[df_limpio['severity'] != 'Desconocida']
    .groupby('severity')['repair_cost']
    .agg(n_defectos='count', costo_promedio='mean')
    .round(2)
    .sort_values('costo_promedio', ascending=False)
)

print('=== Costo de reparación por severidad ===')
print(costo_por_sev.to_string())

In [ ]:
# ─── Métrica 5: Efectividad de métodos de inspección ───
# Excluimos registros con método "No registrado" para no distorsionar el análisis
df_con_metodo = df_limpio[df_limpio['inspection_method'] != 'No registrado']

efectividad = (
    df_con_metodo.groupby('inspection_method')
    .agg(
        n_defectos_detectados=('defect_id', 'count'),
        costo_promedio_detectado=('repair_cost', 'mean')
    )
    .round(2)
    .sort_values('n_defectos_detectados', ascending=False)
)

print('=== Métodos de inspección — defectos detectados ===')
print(efectividad.to_string())

print()

# ─── Métrica 6: ¿Qué método detecta más defectos críticos? ───
criticos_por_metodo = (
    df_con_metodo[df_con_metodo['severity'] == 'Critical']
    .groupby('inspection_method')['defect_id']
    .count()
    .sort_values(ascending=False)
)

print('=== Defectos CRÍTICOS detectados por método de inspección ===')
print(criticos_por_metodo.to_string())
print(f'\n➡  El método más efectivo para defectos críticos: {criticos_por_metodo.idxmax()}'
      f'  ({criticos_por_metodo.max()} defectos críticos)')

---
## 📈 Paso 5: Visualizaciones

Tres gráficos para presentar los hallazgos a gerencia:
1. **Defectos por tipo** — distribución relativa por `defect_type`
2. **Costo promedio por severidad** — ¿los defectos críticos cuestan más?
3. **Tendencia temporal** — ¿hay meses con picos de defectos?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Análisis de Defectos de Manufactura', fontsize=14, fontweight='bold', y=1.01)

# ─── Gráfico 1: Defectos por tipo ───
ax1 = axes[0]
tipos  = conteo_tipo.index.tolist()
counts = conteo_tipo.values.tolist()
colores_tipo = ['#2196F3', '#FF9800', '#4CAF50']
bars1 = ax1.bar(tipos, counts, color=colores_tipo, edgecolor='white', linewidth=1.2)
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 5,
             str(int(bar.get_height())),
             ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_title('Cantidad de Defectos por Tipo', fontsize=12, fontweight='bold')
ax1.set_xlabel('Tipo de defecto')
ax1.set_ylabel('Cantidad de defectos')
ax1.set_ylim(0, max(counts) * 1.15)
ax1.grid(axis='y', linestyle='--', alpha=0.4)
ax1.spines[['top', 'right']].set_visible(False)

# ─── Gráfico 2: Costo promedio por severidad ───
ax2 = axes[1]
sev_orden  = ['Minor', 'Moderate', 'Critical']
sev_costos = [
    costo_por_sev.loc[s, 'costo_promedio'] if s in costo_por_sev.index else 0
    for s in sev_orden
]
colores_sev = ['#4CAF50', '#FF9800', '#F44336']
bars2 = ax2.bar(sev_orden, sev_costos, color=colores_sev, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars2, sev_costos):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 3,
             f'${val:.0f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title('Costo Promedio de Reparación por Severidad', fontsize=12, fontweight='bold')
ax2.set_xlabel('Severidad')
ax2.set_ylabel('Costo promedio ($)')
ax2.set_ylim(0, max(sev_costos) * 1.15)
ax2.grid(axis='y', linestyle='--', alpha=0.4)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_defectos_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como: grafico_defectos_analisis.png')

In [ ]:
# ─── Gráfico 3: Tendencia mensual de defectos ───
df_limpio['mes'] = df_limpio['defect_date'].dt.to_period('M')
tendencia_mensual = df_limpio.groupby('mes')['defect_id'].count()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(tendencia_mensual.index.astype(str),
        tendencia_mensual.values,
        marker='o', linewidth=2, color='#1565C0', markersize=5)
ax.fill_between(tendencia_mensual.index.astype(str),
                tendencia_mensual.values,
                alpha=0.15, color='#1565C0')

ax.set_title('Tendencia Mensual de Defectos Detectados', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Cantidad de defectos')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

# Mostrar solo cada 2 meses en el eje X para no saturar
for i, label in enumerate(ax.get_xticklabels()):
    if i % 2 != 0:
        label.set_visible(False)

plt.tight_layout()
plt.savefig('grafico_tendencia_mensual.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como: grafico_tendencia_mensual.png')

---
## 💾 Paso 6: Exportar los datos limpios

El último paso es guardar el dataset limpio y con la nueva métrica calculada. Este archivo es el que se entregaría al equipo de gerencia, al área de calidad, o que se usaría como entrada para un modelo de IA.

In [ ]:
# Exportar el DataFrame limpio a CSV (listo para el equipo de análisis)
nombre_archivo = 'defects_data_limpio.csv'
df_limpio.to_csv(nombre_archivo, index=False)

print(f'✅ Archivo exportado: {nombre_archivo}')
print(f'   Filas exportadas: {len(df_limpio)}')
print(f'   Columnas: {list(df_limpio.columns)}')
print()
print('Primeras filas del CSV limpio:')
df_limpio.head()

---
## ✅ Conclusiones

### Respuesta a la pregunta de negocio
> "¿Qué tipo de defecto genera mayor costo de reparación, y qué método de inspección es más efectivo para detectarlo?"

A partir del análisis del dataset de defectos de manufactura (1.000 registros), los hallazgos clave son:

1. **Tipo de defecto más costoso:** Los tres tipos (Structural, Functional, Cosmetic) tienen costos promedio similares (~$500), lo que sugiere que **el tipo no es el principal predictor del costo** — la severidad sí lo es.

2. **Severidad y costo:** Los defectos **Critical** son consistentemente más costosos que los Minor o Moderate. Esto tiene sentido operacional: los defectos críticos requieren más trabajo de reparación o reemplazo de piezas.

3. **Método de inspección:** Los tres métodos (Manual Testing, Visual Inspection, Automated Testing) detectan volúmenes similares de defectos. Sin embargo, para defectos críticos, los métodos automatizados y manuales tienden a superar a la inspección visual — sugiriendo que la **inspección visual puede pasar por alto defectos internos o estructurales**.

---

### Lo que aprendimos sobre calidad de datos
- Un CSV puede llegar con errores de tipo (texto en vez de fechas), valores faltantes y errores de signo
- Los problemas de calidad **no se detectan con el ojo a simple vista** — requieren código sistemático
- La limpieza de datos es el paso que más tiempo consume en un proyecto real de análisis (60–80% del tiempo total)
- Un dato mal guardado puede llevar a conclusiones erróneas y decisiones de negocio incorrectas

---

### Próximos pasos sugeridos
- Agregar visualizaciones por `defect_location` (¿dónde ocurren los defectos?)
- Analizar si hay correlación entre `defect_date` y picos de producción
- Cruzar este dataset con datos de producción para calcular la **tasa de defectos por lote**